In [ ]:
import cv2  # Qué: importa OpenCV. Cómo: expone el sustractor de fondo (createBackgroundSubtractorMOG2), el umbralado (threshold) y la GUI de captura. Por qué: es la única dependencia necesaria para detectar movimiento en video.

# Objeto sustractor de fondo

In [ ]:
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)  # Qué: crea un objeto sustractor de fondo basado en el algoritmo MOG2 (Mixture of Gaussians). Cómo: modela cada píxel del fondo como una mezcla de distribuciones gaussianas aprendidas de los últimos `history` cuadros (500); `varThreshold` controla qué tan distinto debe ser un píxel de ese modelo para considerarse "primer plano" (a mayor valor, menos sensible); `detectShadows=True` hace que además marque las sombras con un valor de gris intermedio (127) en vez de blanco puro. Por qué: separar fondo estático de objetos en movimiento sin necesitar un cuadro de referencia fijo, adaptándose a cambios graduales de iluminación.

# Aplicar un filtro de umbral para resaltar mejor los objetos en movimiento

In [ ]:
cap = cv2.VideoCapture(0)  # Qué: abre la webcam por defecto (índice 0). Por qué: fuente de video en vivo sobre la que se va a detectar movimiento cuadro a cuadro.

while True:  # Qué: procesa la webcam en vivo, cuadro por cuadro, hasta que el usuario salga. Por qué: la detección de movimiento requiere comparar continuamente el estado actual contra el modelo de fondo acumulado.

    ret, frame = cap.read()  # Qué: captura el siguiente cuadro. Cómo: `ret` indica éxito de lectura y `frame` es la imagen BGR actual.
    
    fgmask = fgbg.apply(frame)  # Qué: aplica el sustractor de fondo al cuadro actual. Cómo: `apply` compara el cuadro contra el modelo de gaussianas y devuelve una máscara en escala de grises donde 255=primer plano (movimiento), 0=fondo y 127=sombra detectada; además actualiza internamente el modelo de fondo con este nuevo cuadro. Por qué: es el mecanismo central de MOG2 para aislar los píxeles que cambiaron respecto al fondo aprendido.

    _, movement = cv2.threshold(fgmask, 254, 255, cv2.THRESH_BINARY)  # Qué: aplica un umbral binario a la máscara de primer plano. Cómo: THRESH_BINARY pone en 255 todo píxel con valor > 254 y en 0 el resto, descartando así el valor 127 (sombras) que devuelve MOG2. Por qué: filtra las sombras para quedarse solo con el movimiento "real", evitando falsos positivos por cambios de iluminación proyectada.


    cv2.imshow("Cuadro Original", frame)  # Qué: muestra el video original sin procesar. Por qué: sirve de referencia visual para comparar contra la máscara de movimiento.
    cv2.imshow("Deteccion de Movimiento", movement)  # Qué: muestra la máscara binaria resultante (blanco = movimiento detectado, negro = fondo/sombra). Por qué: visualiza el resultado final del pipeline de detección.

    if cv2.waitKey(1) == ord('q'):  # Qué: espera 1 ms por una tecla (y procesa eventos de GUI). Cómo: compara el código de tecla contra el de 'q'. Por qué: permite salir del loop de forma controlada por el usuario.
        break  # Qué: interrumpe el bucle infinito. Por qué: única salida limpia del loop de captura.
    
cap.release()  # Qué: libera el dispositivo de cámara. Por qué: evita que la webcam quede bloqueada para otros procesos.
cv2.destroyAllWindows()  # Qué: cierra todas las ventanas abiertas por OpenCV. Por qué: limpieza de recursos de GUI al terminar.


## 🧪 Práctica
Reforzá lo aprendido en este módulo resolviendo los ejercicios guiados en [`practicas/6_practica.ipynb`](../practicas/6_practica.ipynb).